In [4]:
import pandas as pd
import os
from tasks.classification import parse_class_to_dataset

In [5]:
# Path to the per_class.csv file
output_dir = "/group/jmearlesgrp/intermediate_data/eranario/vlm-investigation/fine_tune_classification/gemma_3/fold_1_train17_val10"  # Change this to your output directory
per_class_csv = os.path.join(output_dir, "per_class.csv")

# Load per-class metrics
df = pd.read_csv(per_class_csv)
print(f"Loaded {len(df)} classes from {per_class_csv}")
df.head()

Loaded 88 classes from /group/jmearlesgrp/intermediate_data/eranario/vlm-investigation/fine_tune_classification/gemma_3/fold_1_train17_val10/per_class.csv


,class,precision,recall,f1,support
0,crop_weeds_greece-black_nightshade,0.000000,0.0,0.000000,25.0
1,crop_weeds_greece-cotton,0.000546,1.0,0.001092,11.0
2,crop_weeds_greece-tomato,0.000000,0.0,0.000000,41.0
3,crop_weeds_greece-velvet_leaf,0.000000,0.0,0.000000,26.0
4,papaya_leaf_disease_classification-Anthracnose,0.000000,0.0,0.000000,71.0


In [6]:
# Parse dataset name from class name
df['dataset'], df['class_name'] = zip(*df['class'].apply(parse_class_to_dataset))

# Show the parsed data
print(f"\nFound {df['dataset'].nunique()} unique datasets:")
print(df['dataset'].unique())
df.head(10)


Found 10 unique datasets:
['crop_weeds_greece' 'papaya_leaf_disease_classification' 'plant_village_classification' 'rice_leaf_disease_classification' 'soybean_weed_uav_brazil' 'sugarcane_damage_usa' 'sunflower_disease_classification' 'tea_leaf_disease_classification' 'tomato_leaf_disease' 'vine_virus_photo_dataset']


,class,precision,recall,f1,support,dataset,class_name
0,crop_weeds_greece-black_nightshade,0.000000,0.0,0.000000,25.0,crop_weeds_greece,black_nightshade
1,crop_weeds_greece-cotton,0.000546,1.0,0.001092,11.0,crop_weeds_greece,cotton
2,crop_weeds_greece-tomato,0.000000,0.0,0.000000,41.0,crop_weeds_greece,tomato
3,crop_weeds_greece-velvet_leaf,0.000000,0.0,0.000000,26.0,crop_weeds_greece,velvet_leaf
4,papaya_leaf_disease_classification-Anthracnose,0.000000,0.0,0.000000,71.0,papaya_leaf_disease_classification,Anthracnose
5,papaya_leaf_disease_classification-BacterialSpot,0.000000,0.0,0.000000,92.0,papaya_leaf_disease_classification,BacterialSpot
6,papaya_leaf_disease_classification-Curl,0.000000,0.0,0.000000,117.0,papaya_leaf_disease_classification,Curl
7,papaya_leaf_disease_classification-Healthy,0.000000,0.0,0.000000,46.0,papaya_leaf_disease_classification,Healthy
8,papaya_leaf_disease_classification-RingSpot,0.000000,0.0,0.000000,107.0,papaya_leaf_disease_classification,RingSpot
9,plant_village_classification-Apple___Apple_scab,0.000000,0.0,0.000000,126.0,plant_village_classification,Apple___Apple_scab


In [7]:
# Aggregate metrics by dataset
# We'll compute:
# - Average precision, recall, f1 across classes in each dataset
# - Total support (number of samples) per dataset
# - Number of classes per dataset
# - Accuracy per dataset (weighted by support)

# First, compute weighted accuracy per dataset
def compute_dataset_accuracy(group):
    """Compute accuracy as weighted average of per-class recall by support"""
    total_support = group['support'].sum()
    if total_support == 0:
        return 0.0
    weighted_recall = (group['recall'] * group['support']).sum() / total_support
    return weighted_recall

dataset_metrics = df.groupby('dataset').agg({
    'precision': 'mean',
    'recall': 'mean',
    'f1': 'mean',
    'support': 'sum',
    'class': 'count'  # counts number of classes
}).rename(columns={'class': 'n_classes'})

# Add accuracy as weighted average
dataset_metrics['accuracy'] = df.groupby('dataset').apply(compute_dataset_accuracy)

# Reorder columns
dataset_metrics = dataset_metrics[['n_classes', 'support', 'accuracy', 'precision', 'recall', 'f1']]

# Round to 4 decimal places
dataset_metrics = dataset_metrics.round(4)

# Sort by dataset name
dataset_metrics = dataset_metrics.sort_index()

print(f"\nDataset-level metrics:")
dataset_metrics


Dataset-level metrics:


/tmp/ipykernel_265214/2310309574.py:26: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dataset_metrics['accuracy'] = df.groupby('dataset').apply(compute_dataset_accuracy)


,n_classes,support,accuracy,precision,recall,f1
dataset,,,,,,
crop_weeds_greece,4,103.0,0.1068,0.0001,0.25,0.0003
papaya_leaf_disease_classification,5,433.0,0.0000,0.0000,0.00,0.0000
plant_village_classification,39,11105.0,0.0000,0.0000,0.00,0.0000
rice_leaf_disease_classification,6,769.0,0.0000,0.0000,0.00,0.0000
soybean_weed_uav_brazil,4,3069.0,0.0000,0.0000,0.00,0.0000
sugarcane_damage_usa,6,33.0,0.0000,0.0000,0.00,0.0000
sunflower_disease_classification,4,472.0,0.0000,0.0000,0.00,0.0000
tea_leaf_disease_classification,6,1174.0,0.0000,0.0000,0.00,0.0000
tomato_leaf_disease,10,2200.0,0.0000,0.0000,0.00,0.0000


In [8]:
# Save to CSV
dataset_metrics_csv = os.path.join(output_dir, "dataset_metrics.csv")
dataset_metrics.to_csv(dataset_metrics_csv)

print(f"\nSaved dataset metrics to: {dataset_metrics_csv}")


Saved dataset metrics to: /group/jmearlesgrp/intermediate_data/eranario/vlm-investigation/fine_tune_classification/gemma_3/fold_1_train17_val10/dataset_metrics.csv


In [9]:
# Optional: Show summary statistics
print("\nSummary Statistics Across All Datasets:")
print(f"Total datasets: {len(dataset_metrics)}")
print(f"Total samples: {dataset_metrics['support'].sum()}")
print(f"Total classes: {dataset_metrics['n_classes'].sum()}")
print(f"\nAverage metrics across datasets:")
print(f"  Accuracy: {dataset_metrics['accuracy'].mean():.4f}")
print(f"  Precision: {dataset_metrics['precision'].mean():.4f}")
print(f"  Recall: {dataset_metrics['recall'].mean():.4f}")
print(f"  F1 Score: {dataset_metrics['f1'].mean():.4f}")


Summary Statistics Across All Datasets:
Total datasets: 10
Total samples: 20133.0
Total classes: 88

Average metrics across datasets:
  Accuracy: 0.0107
  Precision: 0.0000
  Recall: 0.0250
  F1 Score: 0.0000
